<div align="center">
  <p>
    <a href="https://ultralytics.com" target="_blank">
      <img width="100%" src="https://raw.githubusercontent.com/ultralytics/assets/main/yolov8/banner-yolov8.png" alt="Ultralytics YOLO banner">
    </a>
  </p>

[![Ultralytics Actions](https://github.com/ultralytics/ultralytics/actions/workflows/ci.yaml/badge.svg)](https://github.com/ultralytics/ultralytics/actions/workflows/ci.yaml)
[![Ultralytics Discord](https://img.shields.io/discord/1089800235347353640?logo=discord&logoColor=white&label=Discord&color=blue)](https://ultralytics.com/discord)
[![Ultralytics Forums](https://img.shields.io/discourse/users?server=https%3A%2F%2Fcommunity.ultralytics.com&logo=discourse&label=Forums&color=blue)](https://community.ultralytics.com)
[![Ultralytics Reddit](https://img.shields.io/reddit/subreddit-subscribers/ultralytics?style=flat&logo=reddit&logoColor=white&label=Reddit&color=blue)](https://reddit.com/r/ultralytics)

Welcome to the Ultralytics YOLO26 notebook for **Autonomous Vehicle Object Detection**! This notebook demonstrates real-time detection of vehicles, pedestrians, traffic lights, and road obstacles using the latest YOLO26 model — purpose-built for edge deployment with NMS-free end-to-end inference.

📚 **Documentation**: https://docs.ultralytics.com/models/yolo26  
💬 **Discord**: https://ultralytics.com/discord
</div>

## 1. 📦 Install Dependencies

In [ ]:
# Install Ultralytics package
%pip install ultralytics supervision requests -q

import ultralytics
ultralytics.checks()

## 2. 🚗 Load YOLO26 Model

YOLO26 introduces **NMS-free end-to-end inference** and the **MuSGD optimizer**, making it 43% faster on CPU compared to previous versions — ideal for autonomous driving edge hardware.

In [ ]:
from ultralytics import YOLO

# Load YOLO26 nano model (fastest, best for edge deployment)
model = YOLO("yolo26n.pt")

print(f"Model: {model.info()}")
print(f"\nClasses relevant to autonomous driving:")
adas_classes = {
    k: v for k, v in model.names.items()
    if v in ["person", "bicycle", "car", "motorcycle", "bus", "truck",
             "traffic light", "stop sign", "fire hydrant"]
}
for idx, name in adas_classes.items():
    print(f"  [{idx}] {name}")

## 3. 🖼️ Download Sample Driving Images

We'll use publicly available driving scene images for inference.

In [ ]:
import requests
import os
from pathlib import Path

# Create directory for sample images
Path("driving_samples").mkdir(exist_ok=True)

# Sample driving scene image URLs (COCO / open license)
sample_urls = {
    "urban_scene.jpg": "https://ultralytics.com/images/bus.jpg",
    "street_scene.jpg": "https://ultralytics.com/images/zidane.jpg",
}

for filename, url in sample_urls.items():
    path = f"driving_samples/{filename}"
    if not os.path.exists(path):
        response = requests.get(url)
        with open(path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: {filename}")

print("\nSample images ready!")

## 4. 🔍 Run Inference — Detect ADAS Objects

We filter detections to only show classes relevant to autonomous driving: **vehicles, pedestrians, traffic lights, and stop signs**.

In [ ]:
from ultralytics import YOLO
import glob

model = YOLO("yolo26n.pt")

# COCO class indices for ADAS-relevant objects
ADAS_CLASSES = [
    0,   # person
    1,   # bicycle
    2,   # car
    3,   # motorcycle
    5,   # bus
    7,   # truck
    9,   # traffic light
    11,  # stop sign
]

# Run inference on all sample images
image_paths = glob.glob("driving_samples/*.jpg")

results = model.predict(
    source=image_paths,
    classes=ADAS_CLASSES,   # Filter to ADAS classes only
    conf=0.35,              # Confidence threshold
    iou=0.5,                # IoU threshold
    save=True,              # Save annotated images
    project="adas_output",
    name="yolo26_inference",
    exist_ok=True,
)

# Print detection summary
for r in results:
    print(f"\nImage: {r.path}")
    print(f"  Detections: {len(r.boxes)}")
    if len(r.boxes) > 0:
        for box in r.boxes:
            cls_name = model.names[int(box.cls)]
            conf = float(box.conf)
            print(f"  → {cls_name}: {conf:.2f}")

## 5. 📊 Visualize Results

In [ ]:
from IPython.display import display, Image as IPImage
import glob

# Display all saved output images
output_images = glob.glob("adas_output/yolo26_inference/*.jpg")

for img_path in output_images:
    print(f"Result: {img_path}")
    display(IPImage(img_path, width=800))

## 6. 📹 Real-Time Inference on Video

YOLO26's NMS-free architecture makes it ideal for real-time video. Upload your own dashcam footage or use the sample below.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

# Option 1: Use a YouTube dashcam video
# Option 2: Upload your own video and replace the path below
VIDEO_SOURCE = "https://youtu.be/LNwODJXcvt4"  # Replace with your dashcam video

results = model.predict(
    source=VIDEO_SOURCE,
    classes=ADAS_CLASSES,
    conf=0.4,
    save=True,
    project="adas_output",
    name="yolo26_video",
    exist_ok=True,
    stream=True,   # Stream results frame by frame (memory efficient)
)

frame_count = 0
for r in results:
    frame_count += 1
    if frame_count % 30 == 0:  # Print every 30 frames
        print(f"Frame {frame_count}: {len(r.boxes)} detections")

print(f"\nTotal frames processed: {frame_count}")
print("Video saved to adas_output/yolo26_video/")

## 7. ⚡ Benchmark YOLO26 vs YOLO11

Compare inference speed across model sizes — key for choosing the right model for edge hardware.

In [ ]:
from ultralytics import YOLO
import time
import numpy as np

MODELS = {
    "YOLO26n": "yolo26n.pt",
    "YOLO11n": "yolo11n.pt",
    "YOLO26s": "yolo26s.pt",
    "YOLO11s": "yolo11s.pt",
}

test_image = "driving_samples/urban_scene.jpg"
RUNS = 20  # Number of inference runs for averaging

print(f"{'Model':<12} {'Avg Latency (ms)':<20} {'FPS':<10}")
print("-" * 42)

for model_name, model_path in MODELS.items():
    model = YOLO(model_path)
    # Warmup
    model.predict(test_image, verbose=False)

    times = []
    for _ in range(RUNS):
        start = time.perf_counter()
        model.predict(test_image, verbose=False)
        times.append((time.perf_counter() - start) * 1000)

    avg_ms = np.mean(times)
    fps = 1000 / avg_ms
    print(f"{model_name:<12} {avg_ms:<20.1f} {fps:<10.1f}")

## 8. 🏋️ Fine-Tune on Custom Driving Dataset

Fine-tune YOLO26 on your own dashcam or annotated driving dataset. The example below uses the built-in `coco8.yaml` as a placeholder — replace with your own dataset YAML.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")  # Start from pretrained YOLO26n

# Replace 'coco8.yaml' with your custom dataset YAML
# Your YAML should define: path, train, val, nc (num classes), names
results = model.train(
    data="coco8.yaml",   # 🔁 Replace with your dataset YAML
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,            # GPU (use 'cpu' if no GPU)
    project="adas_training",
    name="yolo26n_adas",
    exist_ok=True,
    patience=10,         # Early stopping
    save=True,
)

print(f"Training complete!")
print(f"Best model saved to: {results.save_dir}")

## 9. 📤 Export for Edge Deployment

YOLO26's NMS-free design simplifies export — no post-processing needed on device.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

# Export to ONNX (works on most edge hardware)
model.export(format="onnx", imgsz=640, simplify=True)
print("Exported to ONNX")

# Export to TensorRT (NVIDIA Jetson / GPU edge devices)
# model.export(format="engine", imgsz=640, half=True)

# Export to TFLite (mobile / Raspberry Pi)
# model.export(format="tflite", imgsz=640, int8=True)

## 10. 📚 What's Next?

- 📖 [YOLO26 Documentation](https://docs.ultralytics.com/models/yolo26)
- 🚀 [Ultralytics Platform](https://www.ultralytics.com/)
- 💬 [Community Discord](https://ultralytics.com/discord)
- 🐛 [Report Issues](https://github.com/ultralytics/ultralytics/issues)
- 🤝 [Contributing Guide](https://docs.ultralytics.com/help/contributing)

---

<div align="center">
  Built with ❤️ by the Ultralytics community
</div>